In [1]:
# Standard libraries
import os
from pathlib import Path
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [2]:
from __future__ import annotations
import os
import math
import pandas as pd
from typing import List, Tuple, Dict, Optional

def evaluar_contra_modelos(
    df_nuevo: pd.DataFrame,
    modelos: List[Tuple[str, float]],
    *,
    id_col: str = "SamplingOperations_code",
    pred_col: str = "IBD_EQR_Status",
    carpeta: Optional[str] = None,  # si todos están en la misma carpeta, puedes pasarla aquí
    candidatos_pred_cols: Optional[List[str]] = None,  # override si ya conoces el nombre exacto
    normalizar_labels: bool = True,
) -> Dict[str, object]:
    """
    Compara las predicciones de `df_nuevo` contra varios modelos ya evaluados (guardados como CSV)
    y contra consensos (mayoría simple y mayoría ponderada por accuracies). También estima la
    probabilidad esperada de que la predicción de `df_nuevo` sea correcta.

    Parámetros
    ----------
    df_nuevo : DataFrame
        Debe contener [id_col, pred_col] con las predicciones a evaluar.
    modelos : List[Tuple[str, float]]
        Lista de (ruta_csv, accuracy) para cada modelo histórico evaluado.
        - 'accuracy' debe venir en [0,1]. Si la tienes en %, pásala/convierte a proporción.
    id_col : str
        Nombre de la columna id. Default: "SamplingOperations_code".
    pred_col : str
        Nombre de la columna de la predicción de df_nuevo. Default: "IBD_EQR_Status".
    carpeta : str | None
        Carpeta base donde están los CSV (si la ruta de cada CSV no es absoluta).
    candidatos_pred_cols : list[str] | None
        Lista de posibles nombres de columna de predicción dentro de cada CSV.
        Si None, se intenta autodetección inteligente.
    normalizar_labels : bool
        Si True, hace strip y upper() en labels string para robustez.

    Retorna
    -------
    dict con:
      - 'resumen_modelos': DataFrame (modelo, accuracy, n_overlap, coincidencia, archivo)
      - 'acuerdo_consenso_simple': float
      - 'acuerdo_consenso_ponderado': float
      - 'esperado_correcto_promedio': float
      - 'detallado': DataFrame con columnas:
            [id_col, pred_col, <preds de cada modelo>, consenso_simple, consenso_pond,
             prob_correcta_estimada, match_<modelo>...]
    """
    # --- Validaciones básicas
    if id_col not in df_nuevo.columns or pred_col not in df_nuevo.columns:
        raise ValueError(f"`df_nuevo` debe contener las columnas '{id_col}' y '{pred_col}'.")

    df_base = df_nuevo[[id_col, pred_col]].copy()

    # Normalización suave de labels para evitar mismatches por casing/espacios
    def _norm(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            s = x.strip()
            return s.upper() if normalizar_labels else s
        return x

    if normalizar_labels:
        df_base[pred_col] = df_base[pred_col].map(_norm)

    # Heurística de autodetección del nombre de columna de predicción
    default_candidates = [
        pred_col,
        "IBD_EQR_Status",
        "prediction",
        "pred",
        "status",
    ]
    # Permitimos columnas con prefijo, p.ej. IBD_EQR_Status_81
    def _find_pred_col(cols: List[str]) -> str:
        cands = (candidatos_pred_cols or default_candidates)
        # 1) coincidencia exacta por candidatos
        for c in cands:
            if c in cols:
                return c
        # 2) heurística: la que empiece con 'IBD_EQR_Status'
        for c in cols:
            if str(c).startswith("IBD_EQR_Status"):
                return c
        # 3) fallback: escoger la primera no-id con dtype 'object' o 'category'
        for c in cols:
            if c != id_col:
                return c
        raise ValueError("No pude detectar la columna de predicción del CSV del modelo.")

    # Cargar predicciones de modelos y armar tabla combinada
    tablas = []
    resumen_rows = []
    nombre_cols_modelo = []

    for ruta, acc in modelos:
        if acc > 1.0:  # si viene en %, conviértelo
            acc = acc / 100.0

        archivo = os.path.join(carpeta, ruta) if (carpeta and not os.path.isabs(ruta)) else ruta
        df_m = pd.read_csv(archivo)

        if id_col not in df_m.columns:
            raise ValueError(f"En '{archivo}' no existe la columna id '{id_col}'.")

        col_pred_m = _find_pred_col(df_m.columns.tolist())
        col_modelo = os.path.splitext(os.path.basename(archivo))[0]  # nombre corto desde el filename
        col_modelo_pred = f"pred__{col_modelo}"  # evitar colisiones

        tmp = df_m[[id_col, col_pred_m]].rename(columns={col_pred_m: col_modelo_pred})

        if normalizar_labels:
            tmp[col_modelo_pred] = tmp[col_modelo_pred].map(_norm)

        tablas.append(tmp)
        nombre_cols_modelo.append((col_modelo, col_modelo_pred, acc, archivo))

    # Merge incremental por id
    df_all = df_base.copy()
    for _, col_pred_m, _, _ in nombre_cols_modelo:
        # merge secuencial (left) para conservar todos los ids de df_nuevo
        df_all = df_all.merge(
            next(t for t in tablas if col_pred_m in t.columns),
            on=id_col, how="left"
        )

    # Coincidencia por modelo
    for col_modelo, col_pred_m, acc, archivo in nombre_cols_modelo:
        match_col = f"match__{col_modelo}"
        df_all[match_col] = (df_all[pred_col] == df_all[col_pred_m]) & df_all[pred_col].notna() & df_all[col_pred_m].notna()

        overlap = df_all[col_pred_m].notna().sum()
        coinc = df_all[match_col].mean() if overlap > 0 else float("nan")

        resumen_rows.append({
            "modelo": col_modelo,
            "accuracy_reportada": acc,
            "n_overlap": int(overlap),
            "coincidencia_con_df": float(coinc) if not math.isnan(coinc) else None,
            "archivo": archivo
        })

    resumen_modelos = pd.DataFrame(resumen_rows)

    # Consenso (mayoría simple: sin ponderar)
    cols_modelos_pred = [c for _, c, _, _ in nombre_cols_modelo]
    if len(cols_modelos_pred) == 0:
        raise ValueError("No se proporcionaron modelos para comparar.")

    def _mode_row(row_vals):
        # mode() fila a fila (ignorando NaNs). Si hay empate, devuelve el primero por orden.
        s = pd.Series(row_vals).dropna()
        if s.empty:
            return pd.NA
        return s.mode().iloc[0]

    df_all["consenso_simple"] = df_all[cols_modelos_pred].apply(_mode_row, axis=1)

    # Consenso ponderado por accuracies: etiqueta con suma de pesos (accuracies) más alta
    etiqueta_unica = set()
    # recolectar universo de etiquetas posibles (para robustez)
    for c in cols_modelos_pred:
        etiqueta_unica.update(df_all[c].dropna().unique().tolist())

    etiquetas = list(etiqueta_unica)

    pesos = {col_pred_m: acc for _, col_pred_m, acc, _ in nombre_cols_modelo}

    def _consenso_ponderado(row):
        # suma de accuracies por etiqueta propuesta
        best_label, best_w = (pd.NA, -1.0)
        for label in etiquetas:
            w = 0.0
            for col in cols_modelos_pred:
                val = row[col]
                if pd.isna(val): 
                    continue
                if val == label:
                    w += pesos[col]
            if w > best_w:
                best_label, best_w = label, w
        return best_label

    df_all["consenso_pond"] = df_all.apply(_consenso_ponderado, axis=1)

    # Acuerdo df vs consensos
    acuerdo_simple = (df_all[pred_col] == df_all["consenso_simple"]) & df_all[pred_col].notna() & df_all["consenso_simple"].notna()
    acuerdo_ponderado = (df_all[pred_col] == df_all["consenso_pond"]) & df_all[pred_col].notna() & df_all["consenso_pond"].notna()

    acuerdo_consenso_simple = acuerdo_simple.mean()
    acuerdo_consenso_ponderado = acuerdo_ponderado.mean()

    # Estimación de prob. de estar correcto por registro
    # Heurística: prob_correcta(id) = (suma de accuracies de modelos que coinciden con df_nuevo) / (suma de accuracies disponibles)
    sum_acc_total = sum(pesos.values()) if len(pesos) else 0.0

    def _prob_correcta_estimada(row):
        if pd.isna(row[pred_col]) or sum_acc_total == 0:
            return pd.NA
        agree_w = 0.0
        any_obs = False
        for col in cols_modelos_pred:
            val = row[col]
            if pd.isna(val): 
                continue
            any_obs = True
            if val == row[pred_col]:
                agree_w += pesos[col]
        if not any_obs:
            return pd.NA
        return agree_w / sum_acc_total

    df_all["prob_correcta_estimada"] = df_all.apply(_prob_correcta_estimada, axis=1)
    esperado_correcto_promedio = float(df_all["prob_correcta_estimada"].dropna().mean()) if df_all["prob_correcta_estimada"].notna().any() else None

    # Orden final de columnas “bonitas”
    orden_cols = [id_col, pred_col] + cols_modelos_pred + [f"match__{m}" for m, _, _, _ in nombre_cols_modelo] + ["consenso_simple", "consenso_pond", "prob_correcta_estimada"]
    detallado = df_all[orden_cols].copy()

    return {
        "resumen_modelos": resumen_modelos.sort_values("modelo").reset_index(drop=True),
        "acuerdo_consenso_simple": float(acuerdo_consenso_simple) if acuerdo_consenso_simple == acuerdo_consenso_simple else None,
        "acuerdo_consenso_ponderado": float(acuerdo_consenso_ponderado) if acuerdo_consenso_ponderado == acuerdo_consenso_ponderado else None,
        "esperado_correcto_promedio": esperado_correcto_promedio,
        "detallado": detallado
    }


In [3]:
# tell python it is runnig in results\compare
os.chdir(os.path.join("results", "compare"))


df_nuevo = pd.read_csv('Boosted_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo
print(salida["acuerdo_consenso_simple"])  # % acuerdo con mayoría simple
print(salida["acuerdo_consenso_ponderado"]) # % acuerdo con mayoría ponderada
print(salida["esperado_correcto_promedio"]) # estimado de acierto promedio de tu df
display(salida["detallado"])     # por-registro (incluye prob_correcta_estimada)


,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.847960,cat_boost_perh.csv
1,rf_por_region,0.8000,5663,0.842310,rf_por_region.csv
2,un_lolazo_submission,0.8100,5063,0.730532,un_lolazo_submission.csv


0.8552004238036377
0.8543174995585379
0.8074983125838665


,SamplingOperations_code,IBD_EQR_Status,pred__un_lolazo_submission,pred__rf_por_region,pred__cat_boost_perh,match__un_lolazo_submission,match__rf_por_region,match__cat_boost_perh,consenso_simple,consenso_pond,prob_correcta_estimada
0,S05169000_20130911,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
1,S05169000_20200819,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
2,S05172000_20210813,GOOD,GOOD,GOOD,GOOD,True,True,True,GOOD,GOOD,1.000000
3,S05172050_20210824,GOOD,POOR,GOOD,MODERATE,False,True,False,GOOD,MODERATE,0.324873
4,S05172350_20120827,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
5658,S03128500_20120723,MODERATE,MODERATE,MODERATE,MODERATE,True,True,True,MODERATE,MODERATE,1.000000
5659,S03128640_20130718,MODERATE,NaN,MODERATE,MODERATE,False,True,True,MODERATE,MODERATE,0.671066
5660,S03128640_20181011,MODERATE,GOOD,MODERATE,MODERATE,False,True,True,MODERATE,MODERATE,0.671066
5661,S03128732_20210817,MODERATE,MODERATE,MODERATE,GOOD,True,True,False,MODERATE,MODERATE,0.653807


In [4]:
df_nuevo = pd.read_csv('Catboost_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo


,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.853081,cat_boost_perh.csv
1,rf_por_region,0.8000,5663,0.846195,rf_por_region.csv
2,un_lolazo_submission,0.8100,5063,0.727883,un_lolazo_submission.csv


In [5]:
consenso_simple1 = salida["detallado"][["SamplingOperations_code", "consenso_simple"]]
consenso_simple1.to_csv("consenso_simple1.csv", index=False)



In [6]:
consenso_pond1 = salida["detallado"][["SamplingOperations_code", "consenso_pond"]]
consenso_pond1.to_csv("consenso_pond1.csv", index=False)

In [7]:
df_nuevo = pd.read_csv('Catboost_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

df_nuevo.to_csv("Catboost_Classificationr.csv", index=False)

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
    ("consenso_pond1.csv",1),
    ("consenso_simple1.csv",1)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.853081,cat_boost_perh.csv
1,consenso_pond1,1.0000,5663,0.856966,consenso_pond1.csv
2,consenso_simple1,1.0000,5663,0.857849,consenso_simple1.csv
3,rf_por_region,0.8000,5663,0.846195,rf_por_region.csv
4,un_lolazo_submission,0.8100,5063,0.727883,un_lolazo_submission.csv


In [8]:
df_nuevo = pd.read_csv('Boosted_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

df_nuevo.to_csv("Boosted_Classificationr.csv", index=False)

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
    ("consenso_pond1.csv",1),
    ("consenso_simple1.csv",1)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.847960,cat_boost_perh.csv
1,consenso_pond1,1.0000,5663,0.854317,consenso_pond1.csv
2,consenso_simple1,1.0000,5663,0.855200,consenso_simple1.csv
3,rf_por_region,0.8000,5663,0.842310,rf_por_region.csv
4,un_lolazo_submission,0.8100,5063,0.730532,un_lolazo_submission.csv


In [9]:
df_nuevo = pd.read_csv('cb_full_t.csv')



# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.831185,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.839131,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.897228,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.910295,cb_t.csv
4,rf_por_region,0.802489,5663,0.843899,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.766202,un_lolazo_submission.csv


0.849072957896299


# EM

In [20]:
df_nuevo = pd.read_csv('../../results/yhat construction/em/Yhat_Uniform.csv')
df_nuevo = df_nuevo[['SamplingOperations_code','y_hat']]
df_nuevo = df_nuevo.rename(columns={"y_hat": "IBD_EQR_Status"})

In [21]:

modelos = [
    ("un_lolazo_submission.csv", 0.8141),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852),
    ('JAPredictionsXGB_acc=0.816907.csv',0.816907),
    ('XGBTruncatedClean_acc=0.372300.csv',0.372300),
    ('XGBTruncatedRaw_acc=0.4999.csv',0.4999),
    ('Interpolation_acc=0.311300.csv',0.311300)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"][["modelo", "accuracy_reportada", "coincidencia_con_df"]])
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,coincidencia_con_df
0,Boosted_Classificationr,0.820462,0.839838
1,Catboost_Classificationr,0.833695,0.848667
2,Interpolation_acc=0.311300,0.311300,0.311849
3,JAPredictionsXGB_acc=0.816907,0.816907,0.829066
4,XGBTruncatedClean_acc=0.372300,0.372300,0.363588
5,XGBTruncatedRaw_acc=0.4999,0.499900,0.497793
6,cat_boost_perh,0.852459,0.862440
7,cb_t,0.868852,0.870740
8,rf_por_region,0.802489,0.822002
9,un_lolazo_submission,0.814100,0.739891


0.7592280407977855


# Integer Programming

In [ ]:
import pandas as pd

df_nuevo = pd.read_csv('Yhat_IntegerProgramming.csv')
df_nuevo = df_nuevo.rename(columns={"y_hat": "IBD_EQR_Status"})

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.8141),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852),
    ('JAPredictionsXGB_acc=0.816907.csv',0.816907),
    ('XGBTruncatedClean_acc=0.372300.csv',0.372300),
    ('XGBTruncatedRaw_acc=0.4999.csv',0.4999),
    ('Interpolation_acc=0.311300.csv',0.311300)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"][["modelo", "accuracy_reportada", "coincidencia_con_df"]])
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,coincidencia_con_df
0,Boosted_Classificationr,0.820462,0.820413
1,Catboost_Classificationr,0.833695,0.833657
2,Interpolation_acc=0.311300,0.311300,0.311319
3,JAPredictionsXGB_acc=0.816907,0.816907,0.816882
4,XGBTruncatedClean_acc=0.372300,0.372300,0.372241
5,XGBTruncatedRaw_acc=0.4999,0.499900,0.499912
6,cat_boost_perh,0.852459,0.852375
7,cb_t,0.868852,0.868974
8,rf_por_region,0.802489,0.802402
9,un_lolazo_submission,0.814100,0.727883


0.7492304549484673


In [ ]:

df_nuevo = df_nuevo[["y_hat", "SamplingOperations_code"]]
df_nuevo = pd.DataFrame(df_nuevo)
df_nuevo = df_nuevo.rename(columns={"y_hat": "IBD_EQR_Status"})
df_nuevo

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"][["modelo", "accuracy_reportada", "coincidencia_con_df"]])
print(salida['esperado_correcto_promedio'])

In [13]:
df_nuevo = pd.read_csv('missing_0_preds.csv')
df_nuevo

,SamplingOperations_code,IBD_EQR_Status
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Good
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Good
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [14]:



# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5063,0.662453,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5063,0.666996,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5063,0.641912,cat_boost_perh.csv
3,cb_t,0.868852,5063,0.642899,cb_t.csv
4,rf_por_region,0.802489,5063,0.638159,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.625716,un_lolazo_submission.csv


0.6464214594493312


In [15]:

df_nuevo = pd.read_csv('011_Status.csv')
df_nuevo

,SamplingOperations_code,IBD_EQR_Status
0,S02000010_20080811,Moderate
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20160825,Good
4,S02000010_20170703,Good
...,...,...
5658,S06940940_20100708,Moderate
5659,S06940940_20230623,Moderate
5660,S06960950_20160629,High
5661,S06960950_20180719,High


In [16]:


df_nuevo = pd.read_csv('011_Status.csv')

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.838425,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.842663,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.879039,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.891930,cb_t.csv
4,rf_por_region,0.802489,5663,0.827300,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.752428,un_lolazo_submission.csv


0.8399361397853008


In [17]:
df_nuevo = pd.read_csv('012_Status.csv')


# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.849020,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.856083,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.909412,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.919124,cb_t.csv
4,rf_por_region,0.802489,5663,0.852198,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.739184,un_lolazo_submission.csv


0.8560128467926588
